# Regime-Adaptive Statistical Arbitrage — Interactive Discovery Demo

This notebook provides **interactive visualisations** of the pair discovery pipeline:

- **Regime timelines**: HMM-detected market states with coloured overlays
- **Top pairs analysis**: ranked table with p-values, half-life, and stability scores
- **Spread Z-score explorer**: interactive chart with regime bands and ±2σ reference lines
- **Regime comparison heatmap**: pair stats aggregated per regime + HMM transition matrix
- **Cointegration precision**: Engle-Granger p-values, hedge ratio precision, scatter of half-life vs p-value
- **Pair stability scoring**: horizontal bar chart + rolling stats per pair
- **Network graph**: Plotly rendering of the cointegration graph
- **Rolling Sharpe & drawdown**: portfolio performance metrics with KPI annotations

> **Prerequisites**: the Flask backend must be running on `http://localhost:5001`.  
> Start it with: `cd dashboard/backend && python app.py`

## 1. Install & Import Dependencies

In [1]:
!python -m pip install --upgrade "nbformat>=4.2.0"

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import plotly
except ImportError:
    _pip("plotly")

try:
    import ipywidgets
except ImportError:
    _pip("ipywidgets")

try:
    import networkx
except ImportError:
    _pip("networkx")

import warnings
warnings.filterwarnings("ignore")

import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import requests
import math
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import networkx as nx
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import nbformat

DARK_BG  = "#081624"
PAPER_BG = "#0d1f30"
GRID_COL = "rgba(255,255,255,0.05)"
FONT_COL = "#c8dde8"

pio.templates["stat_arb_dark"] = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor=PAPER_BG,
        plot_bgcolor=DARK_BG,
        font=dict(color=FONT_COL, family="Inter, system-ui, sans-serif", size=12),
        xaxis=dict(gridcolor=GRID_COL, showgrid=True, zeroline=False,
                   linecolor="rgba(255,255,255,0.08)"),
        yaxis=dict(gridcolor=GRID_COL, showgrid=True, zeroline=False),
        legend=dict(bgcolor="rgba(0,0,0,0)", bordercolor="rgba(255,255,255,0.1)"),
        margin=dict(l=60, r=20, t=50, b=50),
        colorway=["#63e6be", "#ffd166", "#ff8fa3", "#51cf66", "#748ffc", "#da77f2"],
    )
)
pio.templates.default = "stat_arb_dark"

REGIME_META = {
    0: {"label": "Bull / Low-Vol", "color": "#51cf66", "bg": "rgba(81,207,102,0.15)"},
    1: {"label": "Neutral / Mid-Vol", "color": "#ffd166", "bg": "rgba(255,209,102,0.12)"},
    2: {"label": "Bear / High-Vol", "color": "#ff8fa3", "bg": "rgba(255,143,163,0.18)"},
    3: {"label": "Crisis / Extreme-Vol", "color": "#ff6b6b", "bg": "rgba(255,107,107,0.25)"},
}

print(f"✓ Dependencies loaded — plotly {plotly.__version__} | networkx {nx.__version__}")

✓ Dependencies loaded — plotly 6.6.0 | networkx 3.6.1


## 2. Connect to Backend API

In [4]:
BASE = "http://localhost:5001"

def _get(path, **params):
    try:
        r = requests.get(f"{BASE}{path}", params=params, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        return {"ok": False, "error": str(e)}

def get_health():               return _get("/api/health")
def get_regime():               return _get("/api/regime")
def get_equity():               return _get("/api/equity")
def get_drawdown():             return _get("/api/drawdown")
def get_rolling_sharpe(w=60):   return _get("/api/rolling-sharpe", window=w)
def get_summary():              return _get("/api/summary")
def get_ranked_pairs():         return _get("/api/pairs/ranked")
def get_pairs_by_regime():      return _get("/api/pairs/by-regime")
def get_pair_spread(pid):       return _get(f"/api/pairs/{pid}/spread")
def get_network_graph(regime=None):
    return _get("/api/network-graph") if regime is None else _get("/api/network-graph", regime=regime)

# ── Health status ────────────────────────────────────────────────────────────
health = get_health()
if health.get("ok"):
    has_result = health.get("hasResult", False)
    status_color = "#51cf66" if has_result else "#ffd166"
    status_text  = "✓ Backend online" + (" — discovery result available" if has_result else " — run discovery to populate data")
    display(HTML(f"<div style='font-family:Inter,sans-serif;padding:8px 12px;border-left:3px solid {status_color};background:rgba(0,0,0,0.3);color:{status_color};font-size:13px'>{status_text}</div>"))
else:
    display(HTML("<div style='font-family:Inter,sans-serif;padding:8px 12px;border-left:3px solid #ff6b6b;background:rgba(255,107,107,0.08);color:#ff6b6b;font-size:13px'>✗ Backend offline — start it with: <code>cd dashboard/backend && python app.py</code></div>"))

## 3. Interactive Regime Timeline

Portfolio equity curve with coloured regime bands. Use the **range slider** at the bottom to zoom into any period.
The max-drawdown trough is annotated directly on the chart.

In [5]:
regime_resp = get_regime()
equity_resp = get_equity()
dd_resp     = get_drawdown()

regime_df = pd.DataFrame(regime_resp.get("regime", []))
equity_df = pd.DataFrame(equity_resp.get("equity", []))
dd_df     = pd.DataFrame(dd_resp.get("drawdown", []))

if regime_df.empty or equity_df.empty:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>No backtest data yet — run a backtest first from the dashboard, then re-execute this cell.</div>"))
else:
    regime_df["date"] = pd.to_datetime(regime_df["date"])
    equity_df["date"] = pd.to_datetime(equity_df["date"])

    # Build regime bands
    bands = []
    cur_regime = regime_df["value"].iloc[0]
    cur_start  = regime_df["date"].iloc[0]
    for _, row in regime_df.iterrows():
        if row["value"] != cur_regime:
            bands.append({"start": cur_start, "end": row["date"], "regime": cur_regime})
            cur_regime = row["value"]
            cur_start  = row["date"]
    bands.append({"start": cur_start, "end": regime_df["date"].iloc[-1], "regime": cur_regime})

    fig = go.Figure()

    for band in bands:
        meta = REGIME_META.get(int(band["regime"]), {"color": "#888", "bg": "rgba(128,128,128,0.1)", "label": str(band["regime"])})
        fig.add_vrect(
            x0=band["start"], x1=band["end"],
            fillcolor=meta["bg"], line_width=0,
            annotation_text=meta["label"][:4],
            annotation_position="top left",
            annotation_font_size=9,
            annotation_font_color=meta["color"],
        )

    fig.add_trace(go.Scatter(
        x=equity_df["date"], y=equity_df["value"],
        fill="tozeroy", fillcolor="rgba(99,230,190,0.08)",
        line=dict(color="#63e6be", width=2),
        name="Portfolio Equity",
        hovertemplate="%{x|%Y-%m-%d}<br><b>$%{y:,.0f}</b><extra></extra>",
    ))

    if not dd_df.empty:
        dd_df["date"]  = pd.to_datetime(dd_df["date"])
        dd_df["value"] = pd.to_numeric(dd_df["value"], errors="coerce")
        min_idx = dd_df["value"].idxmin()
        trough  = dd_df.loc[min_idx]
        if trough["value"] < -5:
            eq_idx = equity_df.set_index("date")["value"]
            eq_val = eq_idx.asof(trough["date"]) if not eq_idx.empty else 0
            fig.add_annotation(
                x=trough["date"], y=eq_val,
                text=f"Max DD {trough['value']:.1f}%",
                showarrow=True, arrowhead=2, arrowcolor="#ff6b6b",
                font=dict(color="#ff6b6b", size=11),
                bgcolor="rgba(255,107,107,0.15)", bordercolor="#ff6b6b",
            )

    fig.update_layout(
        title="Portfolio Equity with Market Regime Overlays",
        xaxis_title="Date", yaxis_title="Portfolio Value ($)",
        hovermode="x unified",
        xaxis=dict(rangeslider=dict(visible=True, thickness=0.06, bgcolor=PAPER_BG)),
        height=520,
    )
    fig.show()

    legend_html = " &nbsp;&nbsp; ".join(
        f'<span style="color:{v["color"]};font-weight:600">● {v["label"]}</span>'
        for v in REGIME_META.values()
    )
    display(HTML(f'<div style="font-family:Inter,sans-serif;padding:6px 0;font-size:13px">{legend_html}</div>'))

## 4. Top Pairs Analysis Table

Interactive table of all ranked pairs. Use the **Regime dropdown** to filter pairs active in a specific market regime.
p-values are colour-coded: green < 0.01, yellow < 0.05, red ≥ 0.05.

In [6]:
ranked_resp  = get_ranked_pairs()
regime_resp2 = get_pairs_by_regime()

ranked_pairs_raw = ranked_resp.get("pairs", [])   if ranked_resp.get("ok")  else []
pairs_by_regime  = regime_resp2.get("byRegime", {}) if regime_resp2.get("ok") else {}

if not ranked_pairs_raw:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>No ranked pairs — run discovery from the dashboard first (&#9654; Run Discovery), then re-execute.</div>"))
else:
    ranked_df = pd.DataFrame(ranked_pairs_raw)

    def stability_color(label):
        return {"high": "#51cf66", "medium": "#ffd166", "low": "#ff8fa3"}.get(str(label).lower(), "#98a6b3")

    def pvalue_color(p):
        try:
            v = float(p)
            return "#51cf66" if v < 0.01 else "#ffd166" if v < 0.05 else "#ff8fa3"
        except:
            return "#98a6b3"

    def render_pairs_table(df):
        for col in ["rank","pair_id","score","stability_label","regime_sensitive",
                    "best_half_life_days","best_coint_pvalue","n_regimes_active"]:
            if col not in df.columns:
                df[col] = None
        rows = []
        for _, row in df.head(30).iterrows():
            pv   = row.get("best_coint_pvalue")
            stb  = row.get("stability_label", "—")
            pv_fmt = f"{float(pv):.2e}" if pv is not None else "—"
            hl_fmt = f"{float(row['best_half_life_days']):.1f}" if row.get("best_half_life_days") is not None else "—"
            rs_col = "#ff8fa3" if row.get("regime_sensitive") else "#51cf66"
            rows.append(f"""
              <tr style="border-bottom:1px solid rgba(255,255,255,0.04)">
                <td style="padding:5px 8px;color:#98a6b3">{row.get('rank','—')}</td>
                <td style="padding:5px 8px"><strong style="color:#63e6be">{row.get('pair_id','—')}</strong></td>
                <td style="padding:5px 8px;color:#ffd166">{float(row.get('score',0)):.3f}</td>
                <td style="padding:5px 8px"><span style="color:{stability_color(stb)};font-weight:600">{stb}</span></td>
                <td style="padding:5px 8px;color:{rs_col}">{'Yes' if row.get('regime_sensitive') else 'No'}</td>
                <td style="padding:5px 8px">{hl_fmt}</td>
                <td style="padding:5px 8px;color:{pvalue_color(pv)}">{pv_fmt}</td>
                <td style="padding:5px 8px">{row.get('n_regimes_active','—')}</td>
              </tr>""")
        header_style = "padding:6px 8px;text-align:left;color:#98a6b3;border-bottom:1px solid rgba(99,230,190,0.25);font-size:11px;text-transform:uppercase;letter-spacing:.04em"
        return f"""
        <div style="font-family:Inter,sans-serif;overflow-x:auto">
          <table style="width:100%;border-collapse:collapse;font-size:13px;color:#c8dde8">
            <thead><tr>
              <th style="{header_style}">#</th><th style="{header_style}">Pair</th>
              <th style="{header_style}">Score</th><th style="{header_style}">Stability</th>
              <th style="{header_style}">Regime Sensitive</th><th style="{header_style}">Half-Life (d)</th>
              <th style="{header_style}">Best p-value</th><th style="{header_style}">Active Regimes</th>
            </tr></thead>
            <tbody>{''.join(rows)}</tbody>
          </table>
        </div>"""

    out4 = widgets.Output()
    regime_options4 = [("All Regimes", "")] + [
        (f"{v['label']} ({len(pairs_by_regime.get(str(k),[]))})", str(k))
        for k, v in REGIME_META.items() if str(k) in pairs_by_regime
    ]
    dropdown4 = widgets.Dropdown(
        options=regime_options4, description="Regime:",
        style={"description_width": "60px"}, layout=widgets.Layout(width="320px")
    )

    def on_regime_filter(change):
        with out4:
            clear_output(wait=True)
            key = change["new"]
            if key == "":
                df = ranked_df.copy()
            else:
                raw = pairs_by_regime.get(key, [])
                df  = pd.DataFrame(raw)
                # rename regime-specific cols to canonical names
                for old, new in [("coint_pvalue","best_coint_pvalue"),("half_life_days","best_half_life_days")]:
                    if old in df.columns and new not in df.columns:
                        df = df.rename(columns={old: new})
                # backfill score/stability from ranked_df
                if not df.empty and ("score" not in df.columns or df["score"].isna().all()):
                    score_map = ranked_df.set_index("pair_id")[["score","stability_label"]].to_dict("index") if not ranked_df.empty else {}
                    for c in ["score","stability_label"]:
                        df[c] = df["pair_id"].map(lambda p, col=c: score_map.get(p, {}).get(col))
                df["rank"] = range(1, len(df)+1)
            display(HTML(render_pairs_table(df)))

    dropdown4.observe(on_regime_filter, names="value")
    display(widgets.VBox([dropdown4, out4]))
    on_regime_filter({"new": ""})

## 5. Spread Z-Score Explorer with Regime Overlays

Select any discovered pair from the dropdown. The chart shows the **spread z-score** with:
- Coloured regime bands in the background
- Dashed ±2σ reference lines (entry/exit thresholds)
- Range slider for zooming
- Hedge ratio displayed below the chart

In [7]:
if not ranked_pairs_raw:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>Run discovery first.</div>"))
else:
    pair_ids5 = [p["pair_id"] for p in ranked_pairs_raw[:25]]

    spread_out5 = widgets.Output()
    pair_sel5   = widgets.Dropdown(
        options=pair_ids5,
        value=pair_ids5[0] if pair_ids5 else None,
        description="Pair:",
        style={"description_width": "40px"},
        layout=widgets.Layout(width="280px"),
    )
    hedge_html5 = widgets.HTML(value="")

    def render_spread5(pair_id):
        resp = get_pair_spread(pair_id)
        with spread_out5:
            clear_output(wait=True)
            if not resp.get("ok") or not resp.get("spread"):
                print(f"No spread data available for {pair_id}")
                return
            sp = pd.DataFrame(resp["spread"])
            sp["date"] = pd.to_datetime(sp["date"])

            # Build regime bands
            bands = []
            regime_data = resp.get("regime", [])
            if regime_data:
                rs = pd.DataFrame(regime_data)
                rs["date"] = pd.to_datetime(rs["date"])
                cur = rs["regime"].iloc[0]; start = rs["date"].iloc[0]
                for _, r in rs.iterrows():
                    if r["regime"] != cur:
                        bands.append({"start": start, "end": r["date"], "regime": cur})
                        cur = r["regime"]; start = r["date"]
                bands.append({"start": start, "end": rs["date"].iloc[-1], "regime": cur})

            fig5 = go.Figure()

            for band in bands:
                meta = REGIME_META.get(int(band["regime"]), {"bg": "rgba(128,128,128,0.1)"})
                fig5.add_vrect(x0=band["start"], x1=band["end"], fillcolor=meta["bg"], line_width=0)

            for y_val, label, col in [(2.0,"+2σ","rgba(255,107,107,0.5)"),(-2.0,"−2σ","rgba(255,107,107,0.5)"),(0.0,"0","rgba(255,255,255,0.2)")]:
                fig5.add_hline(
                    y=y_val, line_dash="dash", line_color=col,
                    annotation_text=label, annotation_position="right",
                    annotation_font=dict(color=col, size=10),
                )

            fig5.add_trace(go.Scatter(
                x=sp["date"], y=sp["z"],
                fill="tozeroy", fillcolor="rgba(99,230,190,0.06)",
                line=dict(color="#63e6be", width=1.5), name="Z-score",
                hovertemplate="%{x|%Y-%m-%d}<br>Z = <b>%{y:.4f}</b><extra></extra>",
            ))

            fig5.update_layout(
                title=f"Spread Z-score — {pair_id}",
                xaxis_title="Date", yaxis_title="Z-score",
                hovermode="x unified",
                xaxis=dict(rangeslider=dict(visible=True, thickness=0.05, bgcolor=PAPER_BG)),
                height=430,
            )
            fig5.show()

        hr = resp.get("hedge_ratio")
        hr_str = f"Hedge ratio: <strong style='color:#ffd166'>{float(hr):.4f}</strong>" if hr is not None else ""
        hedge_html5.value = f"<span style='font-family:Inter,sans-serif;font-size:13px;color:#c8dde8;padding:4px 0;display:block'>{hr_str}</span>"

    pair_sel5.observe(lambda c: render_spread5(c["new"]), names="value")
    display(widgets.VBox([pair_sel5, spread_out5, hedge_html5]))
    if pair_ids5:
        render_spread5(pair_ids5[0])

In [16]:
import nbformat; print(nbformat.__version__)

5.10.4


## 6. Regime Comparison Heatmap

Two heatmaps:
1. **Pair statistics per regime** — count, mean p-value, mean half-life, mean correlation
2. **HMM transition probability matrix** — probability of transitioning from each regime to every other regime

In [8]:
if not pairs_by_regime:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>No regime data — run discovery first.</div>"))
else:
    regime_keys    = sorted(pairs_by_regime.keys(), key=lambda x: int(x))
    regime_labels6 = [REGIME_META.get(int(k), {}).get("label", f"Regime {k}") for k in regime_keys]

    # ── Panel A: pair stats per regime ────────────────────────────────────────
    matrix6a = []
    for k in regime_keys:
        rows = pairs_by_regime[k]
        df_r = pd.DataFrame(rows)
        matrix6a.append([
            float(len(rows)),
            df_r["coint_pvalue"].astype(float).mean() if "coint_pvalue" in df_r.columns and not df_r.empty else float("nan"),
            df_r["half_life_days"].astype(float).mean() if "half_life_days" in df_r.columns and not df_r.empty else float("nan"),
            df_r["corr"].astype(float).mean()           if "corr" in df_r.columns and not df_r.empty else float("nan"),
        ])

    mat6a = np.array(matrix6a, dtype=float)
    col_labels6a = ["# Pairs", "Mean p-value", "Mean Half-Life (d)", "Mean Corr"]
    text6a = [[f"{v:.3f}" if not np.isnan(v) else "—" for v in row] for row in mat6a]

    fig6a = go.Figure(go.Heatmap(
        z=mat6a, x=col_labels6a, y=regime_labels6,
        text=text6a, texttemplate="%{text}", textfont=dict(size=13, color="#fff"),
        colorscale=[[0,"#081624"],[0.5,"#1a3a5c"],[1,"#63e6be"]],
        showscale=True, colorbar=dict(thickness=12, len=0.8),
    ))
    fig6a.update_layout(
        title="Pair Statistics by Regime",
        height=300, margin=dict(l=160, t=60),
    )
    fig6a.show()

    # ── Panel B: HMM transition matrix ────────────────────────────────────────
    if not regime_df.empty:
        states6  = sorted(regime_df["value"].unique())
        n6       = len(states6)
        idx6     = {s: i for i, s in enumerate(states6)}
        counts6  = np.zeros((n6, n6))
        vals6    = regime_df["value"].values
        for i in range(1, len(vals6)):
            a, b = idx6[vals6[i-1]], idx6[vals6[i]]
            counts6[a][b] += 1
        row_sums6  = counts6.sum(axis=1, keepdims=True)
        trans_mat6 = np.divide(counts6, row_sums6, where=row_sums6 > 0)
        slabels6   = [REGIME_META.get(int(s), {}).get("label", str(s))[:10] for s in states6]

        fig6b = go.Figure(go.Heatmap(
            z=trans_mat6, x=slabels6, y=slabels6,
            text=[[f"{v:.3f}" for v in row] for row in trans_mat6],
            texttemplate="%{text}", textfont=dict(size=13, color="#fff"),
            colorscale=[[0,"#081624"],[0.5,"rgba(255,143,163,0.5)"],[1,"#ff6b6b"]],
            showscale=True, colorbar=dict(thickness=12, len=0.8),
            zmin=0, zmax=1,
        ))
        fig6b.update_layout(
            title="HMM Regime Transition Probability Matrix",
            xaxis_title="To Regime", yaxis_title="From Regime",
            height=360, margin=dict(l=160, t=60),
        )
        fig6b.show()
    else:
        display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:8px'>Run a backtest to see the HMM transition matrix.</div>"))

## 7. Cointegration Precision & P-Value Analysis

- **Scatter**: half-life vs cointegration p-value, **size ∝ score**, colour ∝ stability label
- **Histogram**: distribution of p-values with α=0.05 and α=0.01 thresholds
- **Summary table**: all pairs with p-value, half-life, hedge ratio, and stability

In [9]:
if not ranked_pairs_raw:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>Run discovery first.</div>"))
else:
    df7 = ranked_df[["pair_id","best_coint_pvalue","best_half_life_days","score","stability_label","n_regimes_active"]].copy()
    for c in ["best_coint_pvalue","best_half_life_days","score"]:
        df7[c] = pd.to_numeric(df7[c], errors="coerce")
    df7 = df7.dropna(subset=["best_coint_pvalue","best_half_life_days","score"])

    color_map7 = {"high": "#51cf66", "medium": "#ffd166", "low": "#ff8fa3"}
    marker_colors = df7["stability_label"].map(color_map7).fillna("#98a6b3")

    # ── Scatter: half-life vs p-value ─────────────────────────────────────────
    fig7a = go.Figure(go.Scatter(
        x=df7["best_half_life_days"],
        y=df7["best_coint_pvalue"],
        mode="markers",
        marker=dict(
            size=np.clip(df7["score"] * 25, 6, 30),
            color=marker_colors,
            line=dict(width=1, color="rgba(0,0,0,0.3)"),
            opacity=0.85,
        ),
        text=df7["pair_id"],
        customdata=np.stack([df7["score"], df7["stability_label"]], axis=-1),
        hovertemplate="<b>%{text}</b><br>Half-life: %{x:.1f}d<br>p-value: %{y:.3e}<br>Score: %{customdata[0]:.3f}<br>Stability: %{customdata[1]}<extra></extra>",
        name="Pairs",
    ))
    fig7a.add_hline(y=0.05, line_dash="dash", line_color="rgba(255,209,102,0.5)",
                    annotation_text="α=0.05", annotation_position="right")
    fig7a.add_hline(y=0.01, line_dash="dash", line_color="rgba(81,207,102,0.5)",
                    annotation_text="α=0.01", annotation_position="right")
    fig7a.update_layout(
        title="Half-Life vs Cointegration p-value  (size = score, colour = stability)",
        xaxis_title="Best Half-Life (days)",
        yaxis_title="Best Coint p-value (log scale)",
        yaxis_type="log",
        height=460,
    )
    # Legend annotations
    for label, color in color_map7.items():
        fig7a.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=10, color=color),
            name=f"Stability: {label}", showlegend=True,
        ))
    fig7a.show()

    # ── Histogram: p-value distribution ───────────────────────────────────────
    fig7b = go.Figure(go.Histogram(
        x=df7["best_coint_pvalue"], nbinsx=30,
        marker_color="#63e6be", opacity=0.75, name="p-values",
        hovertemplate="p-value: %{x:.3f}<br>Count: %{y}<extra></extra>",
    ))
    fig7b.add_vline(x=0.05, line_dash="dash", line_color="#ffd166",
                    annotation_text="α=0.05", annotation_position="top right")
    fig7b.add_vline(x=0.01, line_dash="dash", line_color="#51cf66",
                    annotation_text="α=0.01", annotation_position="top right")
    fig7b.update_layout(
        title="Distribution of Cointegration p-values",
        xaxis_title="p-value", yaxis_title="Count", height=320,
    )
    fig7b.show()

    # ── Summary table ─────────────────────────────────────────────────────────
    tbl7 = df7[["pair_id","best_coint_pvalue","best_half_life_days","score","stability_label","n_regimes_active"]].copy()
    tbl7 = tbl7.rename(columns={
        "pair_id": "Pair", "best_coint_pvalue": "p-value",
        "best_half_life_days": "Half-Life (d)", "score": "Score",
        "stability_label": "Stability", "n_regimes_active": "# Regimes",
    })
    tbl7["p-value"]      = tbl7["p-value"].map(lambda x: f"{x:.4e}")
    tbl7["Half-Life (d)"] = tbl7["Half-Life (d)"].map(lambda x: f"{x:.2f}")
    tbl7["Score"]         = tbl7["Score"].map(lambda x: f"{x:.4f}")

    def _colour_prow(row):
        try:
            p = float(row["p-value"])
            c = "#51cf66" if p < 0.01 else "#ffd166" if p < 0.05 else "#ff8fa3"
        except:
            c = "#98a6b3"
        return [f"color: {c}"] + [""] * (len(row) - 1)

    display(tbl7.style
        .apply(_colour_prow, axis=1)
        .set_table_styles([{"selector": "th", "props": [("text-align","left"),("padding","4px 8px")]}])
        .hide(axis="index")
    )

Pair,p-value,Half-Life (d),Score,Stability,# Regimes
AEP–DE,7.0000e-05,12.30,0.7626,high,2
CL–UNP,0.0000e+00,5.00,0.4000,high,1
FANG–FISV,6.0000e-05,5.50,0.3636,high,1
CAT–ISRG,2.0000e-05,5.50,0.3636,high,1
BKNG–MA,5.8000e-04,5.60,0.3571,high,1
ETN–WMT,6.0000e-05,5.70,0.3509,high,1
CAT–WMT,0.0000e+00,5.80,0.3448,high,1
AAPL–SRE,0.0000e+00,5.80,0.3448,high,1
FISV–MPC,0.0000e+00,5.90,0.3390,high,1
CAT–COST,0.0000e+00,6.00,0.3333,high,1


## 8. Pair Stability Scoring

- **Bar chart**: all pairs sorted by score, coloured by stability label
- **Rolling panel**: select any pair to see its spread z-score and raw spread over time

In [10]:
if not ranked_pairs_raw:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>Run discovery first.</div>"))
else:
    df8 = ranked_df.copy()
    for c in ["score"]:
        df8[c] = pd.to_numeric(df8.get(c, pd.Series(0, index=df8.index)), errors="coerce").fillna(0)

    stab_col_map8 = {"high": "#51cf66", "medium": "#ffd166", "low": "#ff8fa3"}
    df8_sorted = df8.sort_values("score", ascending=True).tail(30)
    bar_colors8 = df8_sorted["stability_label"].map(stab_col_map8).fillna("#98a6b3")

    # ── Horizontal bar chart ──────────────────────────────────────────────────
    fig8a = go.Figure(go.Bar(
        y=df8_sorted["pair_id"],
        x=df8_sorted["score"],
        orientation="h",
        marker=dict(color=bar_colors8, line=dict(width=0)),
        text=df8_sorted["stability_label"],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Score: %{x:.3f}<extra></extra>",
    ))
    # Add invisible legend traces for stability colours
    for label, color in stab_col_map8.items():
        fig8a.add_trace(go.Bar(
            x=[None], y=[None], orientation="h",
            marker=dict(color=color), name=f"Stability: {label}", showlegend=True,
        ))
    fig8a.update_layout(
        title="Pair Stability Scores  (colour = stability_label)",
        xaxis_title="Score",
        margin=dict(l=120),
        height=max(320, len(df8_sorted) * 22),
        barmode="overlay",
    )
    fig8a.show()

    # ── Rolling panel with pair selector ─────────────────────────────────────
    pair_ids8  = df8.sort_values("score", ascending=False)["pair_id"].tolist()
    stab_out8  = widgets.Output()
    pair_sel8  = widgets.Dropdown(
        options=pair_ids8, description="Pair:",
        style={"description_width": "40px"}, layout=widgets.Layout(width="260px")
    )

    def render_stability_panel(pair_id):
        resp = get_pair_spread(pair_id)
        with stab_out8:
            clear_output(wait=True)
            if not resp.get("ok") or not resp.get("spread"):
                print(f"No spread data for {pair_id}")
                return
            sp = pd.DataFrame(resp["spread"])
            sp["date"] = pd.to_datetime(sp["date"])

            sp_titles = ("Spread Z-score", "Spread Value") if "spread" in sp.columns else ("Spread Z-score",)
            rows8 = 2 if "spread" in sp.columns else 1
            fig8b = make_subplots(rows=rows8, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                                  subplot_titles=sp_titles[:rows8])

            fig8b.add_trace(go.Scatter(
                x=sp["date"], y=sp["z"], mode="lines",
                line=dict(color="#63e6be", width=1.2),
                fill="tozeroy", fillcolor="rgba(99,230,190,0.05)",
                name="Z-score",
            ), row=1, col=1)
            fig8b.add_hline(y=2,  line_dash="dash", line_color="rgba(255,107,107,0.4)")
            fig8b.add_hline(y=-2, line_dash="dash", line_color="rgba(255,107,107,0.4)")

            if "spread" in sp.columns:
                fig8b.add_trace(go.Scatter(
                    x=sp["date"], y=sp["spread"], mode="lines",
                    line=dict(color="#ffd166", width=1.2), name="Spread",
                ), row=2, col=1)

            row_data = df8[df8["pair_id"] == pair_id]
            stab  = row_data["stability_label"].values[0]   if not row_data.empty and "stability_label" in row_data else "—"
            hl    = row_data["best_half_life_days"].values[0] if not row_data.empty and "best_half_life_days" in row_data else None
            rs    = row_data["regime_sensitive"].values[0]  if not row_data.empty and "regime_sensitive" in row_data else None

            subtitle = f"Stability: <b style='color:{stab_col_map8.get(str(stab),'#98a6b3')}'>{stab}</b>"
            if hl is not None:
                subtitle += f"  |  Half-life: <b style='color:#ffd166'>{float(hl):.1f}d</b>"
            if rs is not None:
                rs_color = "#ff8fa3" if rs else "#51cf66"
                subtitle += f"  |  Regime Sensitive: <b style='color:{rs_color}'>{'Yes' if rs else 'No'}</b>"

            fig8b.update_layout(
                title=f"Rolling Spread Analysis — {pair_id}",
                height=460, hovermode="x unified",
            )
            fig8b.show()
            display(HTML(f"<div style='font-family:Inter,sans-serif;font-size:13px;padding:4px 0;color:#c8dde8'>{subtitle}</div>"))

    pair_sel8.observe(lambda c: render_stability_panel(c["new"]), names="value")
    display(widgets.VBox([pair_sel8, stab_out8]))
    if pair_ids8:
        render_stability_panel(pair_ids8[0])

## 9. Network Graph of Cointegrated Pairs

Assets as **nodes** (size ∝ degree), cointegrated pairs as **edges** (opacity ∝ weight = 1 − p-value).
Filter by regime using the dropdown. Hover over edges for pair_id, correlation, and weight.

In [11]:
if not ranked_pairs_raw:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>Run discovery first.</div>"))
else:
    def build_network_fig(regime_key=None):
        net_resp = get_network_graph(regime_key)
        if not net_resp.get("ok") or not net_resp.get("nodes"):
            return None, 0
        nodes = net_resp["nodes"]
        edges = net_resp["edges"]

        G = nx.Graph()
        for n in nodes:
            G.add_node(n["id"], label=n.get("label", n["id"]))
        for e in edges:
            G.add_edge(e["source"], e["target"],
                       weight=e.get("weight", 0.5),
                       pair_id=e.get("pair_id",""),
                       corr=e.get("corr", 0))

        node_ids = list(G.nodes)
        n_nodes  = len(node_ids)
        pos = {}
        for i, nid in enumerate(node_ids):
            angle = 2 * math.pi * i / max(n_nodes, 1) - math.pi / 2
            pos[nid] = (math.cos(angle), math.sin(angle))

        max_w = max((d["weight"] for _, _, d in G.edges(data=True)), default=1.0)

        fig9 = go.Figure()

        for u, v, data in G.edges(data=True):
            x0, y0 = pos[u]; x1, y1 = pos[v]
            opacity = 0.18 + 0.72 * (data["weight"] / max_w)
            width   = 1.0  + 2.5  * (data["weight"] / max_w)
            fig9.add_trace(go.Scatter(
                x=[x0, x1, None], y=[y0, y1, None],
                mode="lines",
                line=dict(color=f"rgba(99,230,190,{opacity:.2f})", width=width),
                hoverinfo="text",
                text=f"{data['pair_id']}  corr={float(data['corr']):.3f}  w={float(data['weight']):.3f}",
                showlegend=False,
            ))

        node_x, node_y, node_text, node_sizes, node_hover = [], [], [], [], []
        for nid in node_ids:
            x, y = pos[nid]
            node_x.append(x); node_y.append(y)
            deg = G.degree(nid)
            node_sizes.append(10 + min(deg * 4, 24))
            node_text.append(G.nodes[nid].get("label", nid))
            node_hover.append(f"<b>{nid}</b><br>degree: {deg}")

        fig9.add_trace(go.Scatter(
            x=node_x, y=node_y, mode="markers+text",
            marker=dict(size=node_sizes, color="rgba(99,230,190,0.18)",
                        line=dict(color="#63e6be", width=1.5)),
            text=node_text, textposition="top center",
            textfont=dict(size=10, color="#c8dde8"),
            hovertext=node_hover, hoverinfo="text",
            showlegend=False,
        ))

        regime_label = REGIME_META.get(int(regime_key), {}).get("label", f"Regime {regime_key}") if regime_key is not None else "All Regimes"
        fig9.update_layout(
            title=f"Cointegration Network — {regime_label}",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            height=540, hovermode="closest",
        )
        return fig9, len(edges)

    regime_options9 = [("All Regimes", None)] + [
        (f"{v['label']}", str(k)) for k, v in REGIME_META.items() if str(k) in pairs_by_regime
    ]
    net_dropdown9 = widgets.Dropdown(
        options=regime_options9,
        description="Regime:", style={"description_width": "60px"},
        layout=widgets.Layout(width="260px"),
    )
    net_out9 = widgets.Output()

    def on_net_regime(change):
        with net_out9:
            clear_output(wait=True)
            fig, n_edges = build_network_fig(change["new"])
            if fig:
                fig.show()
                display(HTML(f"<div style='font-family:Inter,sans-serif;font-size:12px;color:#98a6b3;padding:4px 0'>{n_edges} cointegrated pair edges shown. Node size ∝ degree.</div>"))
            else:
                print("No network data for this regime.")

    net_dropdown9.observe(on_net_regime, names="value")
    display(widgets.VBox([net_dropdown9, net_out9]))
    on_net_regime({"new": None})

## 10. Rolling Sharpe & Drawdown Metrics

- **Top panel**: 60-day rolling Sharpe ratio with reference lines at 0 and 1
- **Bottom panel**: drawdown curve (red fill)
- **KPI gauges**: CAGR, Sharpe, Sortino, Max Drawdown, Final Equity
- **Numeric summary**: all statistics to 4 decimal places

> Requires a completed backtest. Run one from the dashboard first.

In [12]:
sharpe_resp10 = get_rolling_sharpe(60)
sharpe_df10   = pd.DataFrame(sharpe_resp10.get("rollingSharpe", [])) if sharpe_resp10.get("ok") else pd.DataFrame()
summary10     = get_summary()
stats10       = summary10.get("stats", {}) if summary10.get("ok") else {}

if equity_df.empty:
    display(HTML("<div style='color:#ffd166;font-family:Inter,sans-serif;padding:12px'>No backtest results — run a backtest from the dashboard first.</div>"))
else:
    # ── Two-panel: Rolling Sharpe + Drawdown ──────────────────────────────────
    fig10 = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
        subplot_titles=("60-Day Rolling Sharpe Ratio", "Drawdown (%)"),
        row_heights=[0.55, 0.45],
    )

    if not sharpe_df10.empty:
        sharpe_df10["date"]  = pd.to_datetime(sharpe_df10["date"])
        sharpe_df10["value"] = pd.to_numeric(sharpe_df10["value"], errors="coerce")
        fig10.add_trace(go.Scatter(
            x=sharpe_df10["date"], y=sharpe_df10["value"],
            mode="lines", line=dict(color="#63e6be", width=1.5),
            fill="tozeroy", fillcolor="rgba(99,230,190,0.06)",
            name="Rolling Sharpe",
            hovertemplate="%{x|%Y-%m-%d}<br>Sharpe: <b>%{y:.4f}</b><extra></extra>",
        ), row=1, col=1)
        fig10.add_hline(y=0, line_dash="dash", line_color="rgba(255,255,255,0.2)", row=1, col=1)
        fig10.add_hline(y=1, line_dash="dot",  line_color="rgba(81,207,102,0.45)",
                        annotation_text="Sharpe=1", annotation_position="right",
                        annotation_font_color="rgba(81,207,102,0.8)", row=1, col=1)

    if not dd_df.empty:
        fig10.add_trace(go.Scatter(
            x=dd_df["date"], y=dd_df["value"],
            mode="lines", fill="tozeroy", fillcolor="rgba(255,107,107,0.12)",
            line=dict(color="#ff6b6b", width=1.2),
            name="Drawdown %",
            hovertemplate="%{x|%Y-%m-%d}<br>DD: <b>%{y:.2f}%</b><extra></extra>",
        ), row=2, col=1)

    fig10.update_layout(
        title="Portfolio Risk & Return Overview",
        height=580, hovermode="x unified",
        xaxis2=dict(rangeslider=dict(visible=True, thickness=0.05, bgcolor=PAPER_BG)),
    )
    fig10.show()

    # ── KPI indicators ────────────────────────────────────────────────────────
    kpis10 = [
        ("CAGR",          stats10.get("cagr_pct"),        "%",  "#51cf66"),
        ("Sharpe Ratio",  stats10.get("sharpe_ratio"),    "",   "#63e6be"),
        ("Sortino Ratio", stats10.get("sortino_ratio"),   "",   "#748ffc"),
        ("Max Drawdown",  stats10.get("max_drawdown_pct"), "%", "#ff8fa3"),
        ("Final Equity",  stats10.get("final_equity"),    "$",  "#ffd166"),
    ]

    valid_kpis = [(n, v, s, c) for n, v, s, c in kpis10 if v is not None]
    if valid_kpis:
        kpi_traces = []
        ncols = len(valid_kpis)
        for idx, (name, val, suffix, color) in enumerate(valid_kpis):
            kpi_traces.append(go.Indicator(
                mode="number",
                value=float(val),
                title=dict(text=name, font=dict(color="#98a6b3", size=13)),
                number=dict(
                    suffix=suffix if suffix != "$" else "",
                    prefix="$" if suffix == "$" else "",
                    font=dict(color=color, size=26),
                    valueformat=",.2f",
                ),
                domain=dict(column=idx),
            ))
        fig_kpi = go.Figure(kpi_traces)
        fig_kpi.update_layout(
            grid=dict(rows=1, columns=ncols, pattern="independent"),
            height=130, margin=dict(l=20, r=20, t=20, b=10),
        )
        fig_kpi.show()

    # ── Numeric summary table ─────────────────────────────────────────────────
    summary_rows10 = [
        ("Initial Capital",  stats10.get("initial_capital")),
        ("Final Equity",     stats10.get("final_equity")),
        ("CAGR %",           stats10.get("cagr_pct")),
        ("Sharpe Ratio",     stats10.get("sharpe_ratio")),
        ("Sortino Ratio",    stats10.get("sortino_ratio")),
        ("Max Drawdown %",   stats10.get("max_drawdown_pct")),
        ("Calmar Ratio",     stats10.get("calmar_ratio")),
        ("Total Trades",     stats10.get("total_trades")),
        ("Win Rate %",       stats10.get("win_rate_pct")),
        ("Avg Trade PnL",    stats10.get("avg_trade_pnl")),
    ]
    tbl10 = pd.DataFrame(
        [(k, f"{float(v):.4f}" if v is not None else "—") for k, v in summary_rows10],
        columns=["Metric", "Value"],
    )
    display(tbl10.style.set_table_styles([
        {"selector": "th", "props": [("text-align","left"),("padding","4px 8px"),("color","#98a6b3")]},
        {"selector": "td", "props": [("padding","4px 8px"),("color","#c8dde8")]},
    ]).hide(axis="index"))

# Regime-Adaptive Statistical Arbitrage — Interactive Discovery Demo

This notebook provides **interactive visualisations** of the pair discovery pipeline:

- **Regime timelines** — HMM-detected market states with coloured overlays
- **Top pairs analysis** — ranked table with p-values, half-life, and stability scores
- **Spread Z-score explorer** — interactive chart with regime bands and ±2σ reference lines
- **Regime comparison heatmap** — pair stats aggregated per regime + HMM transition matrix
- **Cointegration precision** — Engle-Granger p-values, hedge ratio precision, scatter of half-life vs p-value
- **Pair stability scoring** — horizontal bar chart + rolling stats per pair
- **Network graph** — Plotly rendering of the cointegration graph
- **Rolling Sharpe & drawdown** — portfolio performance metrics with KPI annotations

> **Prerequisites**: the Flask backend must be running on `http://localhost:5001`.  
> Start it with: `cd dashboard/backend && python app.py`